# 🧶 Skill Weave — 先玩再学

**Adaptive skill routing for multi-agent systems. 3-stage pipeline. Active learning. Multi-skill weaving.**

> 🚀 点击上方 `Runtime → Run all` 一键体验全部功能。无需安装，无需 API key。

---

## 为什么需要 Skill Weave？

多 Agent 系统有几十上百个技能。当一个新任务到来——"帮我部署到生产环境"——系统怎么知道该用 `deploy` 而不是 `monitor` 或 `ssh`？

关键词匹配 → 技能重叠时崩溃  
静态路由表 → 技能增减后腐烂  
全量 LLM → 141 个技能 = 141 个候选，token 爆炸  

**Skill Weave 的做法：三级过滤 + 在线学习 + 多技能编排。**

In [ ]:
# @title 安装（10秒）
!pip install -q git+https://github.com/Hxh-yaoxing/skill-weave.git 2>/dev/null
!pip install -q pyyaml 2>/dev/null

from skill_weave import SkillRouter, SkillWeave, FeedbackLearner, WeavePlanner, MergeStrategy
print("✅ Skill Weave ready")

---

## 第一幕：基础路由 — 4维评分

注册三个技能，问一个问题，看它怎么选。

In [ ]:
router = SkillRouter()
router.register_skill("deploy",   metadata="deploy services to production server, handle rollback")
router.register_skill("monitor",  metadata="monitor system health metrics, alert on anomalies")
router.register_skill("rollback", metadata="rollback failed deployments to last known good state")

results = router.route("The new deploy broke everything, we need to go back")

for r in results:
    bar = "█" * int(r.score * 20)
    print(f"{r.skill.name:12s} {bar:20s} {r.score:.2f}")
    for dim, val in r.breakdown.items():
        print(f"  └─ {dim}: {val:.2f}")

> 💡 注意：`rollback` 分数最高——虽然 query 里没有 "rollback" 这个词，但语义上最接近。

---

## 第二幕：主动学习 — 让路由自己变聪明

每次路由后记录结果。成功的技能获得加权，失败的被抑制。权重自动调整。

In [ ]:
learner = FeedbackLearner(router)

# 模拟：deploy 失败了两次，rollback 成功了三次
for _ in range(2):
    learner.record("deploy",   "deploy task", success=False, dimension_contributions={"semantic": 0.8, "success": 0.3})
for _ in range(3):
    learner.record("rollback", "deploy task", success=True,  dimension_contributions={"semantic": 0.7, "success": 0.9})

print("📊 学习后的权重变化：")
print(f"   α (semantic): {router.alpha:.3f}  → 语义匹配权重")
print(f"   γ (success):  {router.gamma:.3f}  → 成功率权重 ↑ (因为成功预测了结果)")

stats = learner.stats()
print(f"\n📈 近期成功率: {stats['recent_success_rate']:.0%}")
print(f"   总路由次数: {stats['total_routes']}")

---

## 第三幕：技能编织 — 不止选一个，而是编排一条链

真正的复杂任务需要多个技能协作。注册一条链，自动执行。

In [ ]:
planner = WeavePlanner(router)

# 注册一条 CI/CD 链
planner.register_chain(
    "ci-cd",
    skills=["deploy", "monitor"],
    description="部署后自动监控",
    conditions={1: ("output and 'error' in str(output)", "rollback")},  # 失败则回滚
)

chain = planner.plan("run the ci-cd pipeline")

print(f"🧵 链条: {chain.name}")
print(f"   节点数: {len(chain.nodes)}")
print(f"   依赖边: {len(chain.edges)}")
print()

# 可视化
for node in chain.nodes:
    icon = {"skill": "⚙️", "merge": "🔀", "condition": "🔀"}.get(node.type.value, "?")
    label = node.skill_name or node.type.value
    print(f"   {icon} [{node.type.value.upper():10s}] {label}")

print()
for edge in chain.edges:
    label = f" ({edge.label})" if edge.label else ""
    print(f"   {edge.from_node:30s} → {edge.to_node}{label}")

---

## 第四幕：多链方案 — 一个任务，三条路线

`plan_deep()` 为同一个任务生成多个备选技能序列，让你选择最优路径。

In [ ]:
router2 = SkillRouter()
for name, meta in [
    ("fetch_api", "fetch data from REST API"),
    ("fetch_db", "fetch data from database"),
    ("clean", "clean and normalize raw data"),
    ("analyze", "statistical analysis"),
    ("visualize", "create charts and graphs"),
    ("report", "generate markdown report"),
]:
    router2.register_skill(name, metadata=meta)

planner2 = WeavePlanner(router2)
alternatives = planner2.plan_deep("analyze data and create a report", max_depth=3)

for i, alt in enumerate(alternatives, 1):
    flow = " → ".join(alt)
    print(f"方案 {i}: {flow}")

---

## 核心理念

```
被动路由（传统）:  任务 → 查表 → 选技能 → 结束
Skill Weave:      任务 → 三级过滤 → 主动学习 → 多技能编织 → 反馈闭环 ↩
```

它不是"路由一次就完了"，而是"每次路由都在让它变得更好"。

---

## 下一步

- 📖 [完整文档](https://github.com/Hxh-yaoxing/skill-weave)
- 🐛 [提交 Issue](https://github.com/Hxh-yaoxing/skill-weave/issues)
- ⭐ [Star 支持我们](https://github.com/Hxh-yaoxing/skill-weave)

---

*Built by [FeiMing Studio](https://github.com/Hxh-yaoxing) — where humans and agents build together.*